In [32]:
import os
import asyncpg
import pandas as pd
from dotenv import load_dotenv
import json

load_dotenv("2025-03-10-export-items-app.ipynb.env")
notes_path: str = os.getenv('NOTES_PATH', '')
database_url: str = str(os.getenv('DATABASE_URL', ''))
assert notes_path, "NOTES_PATH is not set"
assert database_url, "DATABASE_URL is not set"

# Load items table from database with schema:
# export const itemTable = pgTable("item", {
#   id: text("id").primaryKey(),
#   name: text("name").unique(),
#   content: text("content").notNull(),
#   contentType: text("content_type").notNull().default("markdown"),
#   dueDate: text("due_date"),
#   createdAt: text("created_at").notNull(),
#   updatedAt: text("updated_at").notNull(),
#   deletedAt: text("deleted_at"),
#   status: text("status"),
#   version: integer("version").notNull().default(0),
#   children: text("children").notNull().default("[]"),
# });
async def fetch_data():
    conn = await asyncpg.connect(database_url)
    records = await conn.fetch('SELECT * FROM item')
    if records:
        log_df = pd.DataFrame(records, columns=records[0].keys())
    else:
        log_df = pd.DataFrame()
    await conn.close()
    return log_df
df = await fetch_data()


In [34]:
df_filtered = df[df['deleted_at'].isna() & (df['content_type'] == "markdown")].sort_values(by="created_at", ascending=False)
metadata_dir = os.path.join(notes_path, '.metadata')
assert os.path.exists(notes_path)
os.makedirs(metadata_dir, exist_ok=True)
for _, row in df_filtered.iterrows():
    filename = f"notes/{row['id']}.md"
    file_path = os.path.join(notes_path, filename)
    content = str(row['content'])
    metadata = {
        'schemaVersion': 1,
        'createdAt': row['created_at'],
        'updatedAt': row['updated_at']
    }
    metadata_path = os.path.join(metadata_dir, filename + '.json')
    os.makedirs(os.path.dirname(metadata_path), exist_ok=True)
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    with open(file_path, 'w') as f:
        f.write(content)